# SAM 3 — thử tách artwork từ ảnh sản phẩm (Colab T4)

Notebook này **chỉ kiểm tra SAM 3 image segmentation** trên ảnh sản phẩm. Không cài ComfyUI, không dựng agent, không dùng model tạo ảnh.

## Trước khi chạy

1. Mở **Runtime → Change runtime type**.
2. Chọn **Hardware accelerator: T4 GPU**.
3. Ở **Runtime Version**, chọn **2026.07 (Python 3.12.13)** nếu tùy chọn này còn hiển thị. Runtime Colab 2026.10 mặc định dùng Python 3.13; bản 2026.07 giúp tránh một số xung đột dependency của stack hiện tại.
4. Trên Hugging Face, đăng nhập và request quyền truy cập model: [facebook/sam3](https://huggingface.co/facebook/sam3). Cần được chấp thuận trước khi tải checkpoint.
5. Tạo Hugging Face **Read** access token tại [Settings → Access Tokens](https://huggingface.co/settings/tokens).

## Notebook sẽ làm gì?

- Cài code chính thức từ `facebookresearch/sam3`.
- Nạp checkpoint SAM 3.
- Upload ảnh mũ/sản phẩm.
- Chạy text prompt để tìm patch/logo/chữ/họa tiết.
- Hiển thị mask, bounding box và confidence score.
- Lưu mask PNG và cutout PNG trong suốt bằng **pixel gốc + mask**, không vẽ lại artwork.

**Lưu ý về T4:** T4 không hỗ trợ BF16 phần cứng; notebook dùng CUDA FP16 autocast khi inference. Chạy từng prompt/ảnh một, không batch để giảm nguy cơ hết VRAM.

## 1. Kiểm tra runtime và GPU

Chạy cell này trước. Nếu không thấy GPU NVIDIA hoặc Python không phải 3.12, hãy kiểm tra lại Runtime Version / Hardware accelerator.

In [ ]:
import sys, torch, platform

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if sys.version_info[:2] != (3, 12):
    print("\nWARNING: Notebook này được chuẩn bị để chạy với Colab Runtime Version 2026.07 (Python 3.12.13).")

if not torch.cuda.is_available():
    raise RuntimeError(
        "Chưa có GPU. Vào Runtime → Change runtime type → Hardware accelerator → T4 GPU, rồi chạy lại cell."
    )

props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0))
print(f"VRAM: {props.total_memory / 1024**3:.1f} GB")

if props.total_memory < 12 * 1024**3:
    print("WARNING: GPU có ít hơn 12GB VRAM; inference có thể gặp CUDA out-of-memory.")


## 2. Cài repository SAM 3 chính thức

Giữ nguyên PyTorch/CUDA đã được Colab cài sẵn. Cell này cài package từ repository chính thức và các dependency của nó.

In [ ]:
import os, subprocess, sys

repo_dir = "/content/sam3"
if not os.path.isdir(repo_dir):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/facebookresearch/sam3.git", repo_dir],
        check=True,
    )

%cd /content/sam3

# Cài SAM 3 theo pyproject.toml của repository; không cài lại torch/torchvision.
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", "."],
    check=True,
)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "matplotlib", "pillow"],
    check=True,
)
print("SAM 3 repository installed.")


## 3. Đăng nhập Hugging Face

SAM 3 checkpoint được lưu trong repo gated trên Hugging Face. Bạn cần:

- Đã request và được cấp quyền truy cập `facebook/sam3`.
- Dùng token loại **Read**.

Token được nhập ẩn, không ghi trực tiếp vào notebook.

In [ ]:
from getpass import getpass
from huggingface_hub import login

hf_token = getpass("Paste Hugging Face READ token (input hidden): ").strip()
if not hf_token:
    raise ValueError("Bạn chưa nhập Hugging Face token.")

login(token=hf_token, add_to_git_credential=False)
del hf_token
print("Hugging Face login completed.")


## 4. Nạp SAM 3 image model

Lần đầu sẽ tải checkpoint từ Hugging Face. Model được nạp lên GPU. Trên T4, notebook dùng FP16 autocast ở bước inference thay vì BF16.

In [ ]:
import gc, time, torch
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

# Ngưỡng confidence để hiển thị các ứng viên. Nếu không tìm thấy đối tượng,
# có thể thử 0.20–0.30 ở phần chỉnh threshold bên dưới.
CONFIDENCE_THRESHOLD = 0.35

if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

start = time.time()
model = build_sam3_image_model(device="cuda", eval_mode=True)
model.eval()
processor = Sam3Processor(
    model,
    device="cuda",
    confidence_threshold=CONFIDENCE_THRESHOLD,
)

print(f"SAM 3 loaded in {time.time() - start:.1f} seconds")
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")


## 5. Upload ảnh sản phẩm

Upload ảnh gốc chiếc mũ (hoặc ảnh sản phẩm muốn test). Nên bắt đầu với ảnh gốc rõ nét, không phải ảnh patch đã tách sẵn.

In [ ]:
from google.colab import files
from PIL import Image
import io, os

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("Chưa upload ảnh.")

image_filename = next(iter(uploaded.keys()))
image = Image.open(io.BytesIO(uploaded[image_filename])).convert("RGB")
image_path = f"/content/{os.path.basename(image_filename)}"
image.save(image_path, quality=95)

print("Image:", image_filename)
print("Size:", image.size)
display(image)


## 6. Chạy text prompt segmentation

Sửa `PROMPT` theo chi tiết muốn tìm. Nên test **một đối tượng mỗi lần** để biết SAM 3 đang tách cái gì.

Các prompt gợi ý cho ảnh mũ:

- `embroidered fabric patch on the front of a baseball cap`
- `embroidered logo on the front of a baseball cap`
- `embroidered mascot patch on the side of a baseball cap`
- `embroidered flower decoration on the visor of a baseball cap`
- `embroidered lettering on a baseball cap`

Bắt đầu bằng prompt lấy toàn bộ patch, bao gồm đường viền vải.

In [ ]:
PROMPT = "embroidered fabric patch on the front of a baseball cap"

# set_image tính visual features một lần; có thể dùng lại khi thử prompt khác.
amp_context = torch.autocast(device_type="cuda", dtype=torch.float16)

with torch.inference_mode(), amp_context:
    image_state = processor.set_image(image)
    output = processor.set_text_prompt(state=image_state, prompt=PROMPT)

masks = output["masks"]
boxes = output["boxes"]
scores = output["scores"]

print("Prompt:", PROMPT)
print("Masks:", tuple(masks.shape))
print("Boxes:", tuple(boxes.shape))
print("Scores:", scores.detach().float().cpu().tolist())
print("GPU memory allocated: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))


## 7. Xem kết quả segmentation

Mỗi ô là một instance mask do SAM 3 trả về. Kiểm tra xem mask có ôm đúng artwork hay bị ăn sang nền / bỏ sót đường viền không.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle


def to_numpy_masks(mask_tensor):
    arr = mask_tensor.detach().float().cpu().numpy()
    # SAM 3 output thường có shape [N, 1, H, W].
    if arr.ndim == 4 and arr.shape[1] == 1:
        arr = arr[:, 0]
    elif arr.ndim == 2:
        arr = arr[None, ...]
    return arr > 0.5


def show_sam3_results(image_pil, result, max_items=12):
    mask_arr = to_numpy_masks(result["masks"])
    score_arr = result["scores"].detach().float().cpu().numpy().reshape(-1)
    box_arr = result["boxes"].detach().float().cpu().numpy()

    n = len(mask_arr)
    print(f"SAM 3 returned {n} instance(s)")
    if n == 0:
        print("Không có mask ở confidence threshold hiện tại. Thử prompt khác hoặc giảm threshold.")
        return

    cols = min(3, n)
    rows = int(np.ceil(min(n, max_items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows), squeeze=False)
    axes = axes.ravel()
    image_np = np.array(image_pil)

    for i in range(min(n, max_items)):
        ax = axes[i]
        ax.imshow(image_np)
        ax.imshow(mask_arr[i], alpha=0.42, cmap="spring", interpolation="nearest")
        if i < len(box_arr):
            x1, y1, x2, y2 = box_arr[i]
            ax.add_patch(Rectangle((x1, y1), x2-x1, y2-y1,
                                   fill=False, edgecolor="yellow", linewidth=1.8))
        score = float(score_arr[i]) if i < len(score_arr) else float("nan")
        ax.set_title(f"Instance {i} | score={score:.3f}")
        ax.axis("off")

    for ax in axes[min(n, max_items):]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_sam3_results(image, output)


## 8. Xuất mask và PNG nền trong suốt

Cell này **không tạo lại/vẽ lại artwork**. Nó giữ pixel RGB gốc, chỉ thay alpha bằng mask SAM 3.

Mỗi instance sẽ có:

- `mask_XX.png`: mask trắng/đen.
- `cutout_XX.png`: artwork đã tách, nền trong suốt.
- `overlay_XX.png`: ảnh gốc có lớp mask màu để dễ review.

Sau đó nén thành ZIP để tải về.

In [ ]:
from PIL import Image as PILImage
import numpy as np
import os, shutil
from IPython.display import display

OUTPUT_DIR = "/content/sam3_test_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

mask_arr = to_numpy_masks(output["masks"])
score_arr = output["scores"].detach().float().cpu().numpy().reshape(-1)
box_arr = output["boxes"].detach().float().cpu().numpy()
source_rgba = np.array(image.convert("RGBA"))

saved = []
for i, mask in enumerate(mask_arr):
    mask_u8 = (mask.astype(np.uint8) * 255)

    # Giữ nguyên pixel gốc; chỉ thay alpha bằng segmentation mask.
    cutout = source_rgba.copy()
    original_alpha = cutout[:, :, 3].astype(np.uint16)
    cutout[:, :, 3] = ((original_alpha * mask_u8.astype(np.uint16)) // 255).astype(np.uint8)

    mask_file = os.path.join(OUTPUT_DIR, f"mask_{i:02d}.png")
    cutout_file = os.path.join(OUTPUT_DIR, f"cutout_{i:02d}.png")
    overlay_file = os.path.join(OUTPUT_DIR, f"overlay_{i:02d}.png")

    PILImage.fromarray(mask_u8, mode="L").save(mask_file)
    PILImage.fromarray(cutout, mode="RGBA").save(cutout_file)

    # Overlay màu chỉ để xem; không dùng làm artwork đầu ra.
    overlay = source_rgba.copy()
    color_layer = np.zeros_like(overlay)
    color_layer[:, :, 0] = 255
    color_layer[:, :, 1] = 40
    color_layer[:, :, 2] = 170
    color_layer[:, :, 3] = (mask.astype(np.uint8) * 110)
    composite = PILImage.alpha_composite(PILImage.fromarray(overlay), PILImage.fromarray(color_layer, mode="RGBA"))
    composite.convert("RGB").save(overlay_file)

    ys, xs = np.where(mask)
    if len(xs):
        pad = 12
        x0, x1 = max(0, xs.min()-pad), min(image.width, xs.max()+pad+1)
        y0, y1 = max(0, ys.min()-pad), min(image.height, ys.max()+pad+1)
        crop_file = os.path.join(OUTPUT_DIR, f"cutout_{i:02d}_crop.png")
        PILImage.fromarray(cutout, mode="RGBA").crop((x0, y0, x1, y1)).save(crop_file)
    else:
        crop_file = None

    score = float(score_arr[i]) if i < len(score_arr) else None
    saved.append({"index": i, "score": score, "mask": mask_file,
                  "cutout": cutout_file, "overlay": overlay_file, "crop": crop_file})

print(f"Saved {len(saved)} instance(s) to {OUTPUT_DIR}")
for item in saved:
    print(item)

# Preview first cutout over checkerboard-like neutral background.
if saved:
    print("Preview cutout 0 (transparent PNG):")
    display(PILImage.open(saved[0]["cutout"]))

zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
print("ZIP:", zip_path)


## 9. Thử lại bằng prompt khác (không cần nạp model lại)

Đổi prompt, chạy cell này, sau đó chạy lại cell **7** và **8** để xem/lưu kết quả mới. Mỗi lần chỉ chạy một prompt.

In [ ]:
# Ví dụ: thay prompt rồi chạy lại cell.
PROMPT = "embroidered mascot patch on the side of a baseball cap"

processor.reset_all_prompts(image_state)
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
    output = processor.set_text_prompt(state=image_state, prompt=PROMPT)

print("Prompt:", PROMPT)
print("Instances:", len(output["scores"]))
print("Scores:", output["scores"].detach().float().cpu().tolist())


## 10. (Tùy chọn) Thử box prompt nếu text prompt chưa tìm được patch

Dùng tọa độ pixel `[x, y, width, height]` của một hình chữ nhật bao quanh patch trên ảnh gốc. Bạn có thể lấy tọa độ bằng cách xem kích thước ảnh và dùng một công cụ xem ảnh có hiển thị cursor coordinates.

SAM 3 nhận box ở dạng normalized `[center_x, center_y, width, height]` trong khoảng 0–1.

In [ ]:
# SỬA 4 GIÁ TRỊ NÀY theo ảnh của bạn: x, y, width, height (pixel).
# Ví dụ minh họa — không phải tọa độ đúng cho mọi ảnh.
BOX_XYWH = [300, 250, 600, 500]

from sam3.model.box_ops import box_xywh_to_cxcywh
from sam3.visualization_utils import normalize_bbox, draw_box_on_image

W, H = image.size
box_xywh = torch.tensor(BOX_XYWH, dtype=torch.float32).view(-1, 4)
box_cxcywh = box_xywh_to_cxcywh(box_xywh)
normalized_box = normalize_bbox(box_cxcywh, W, H).flatten().tolist()

processor.reset_all_prompts(image_state)
with torch.inference_mode(), torch.autocast(device_type="cuda", dtype=torch.float16):
    output = processor.add_geometric_prompt(
        state=image_state,
        box=normalized_box,
        label=True,
    )

print("Box prompt (normalized cx,cy,w,h):", normalized_box)
display(draw_box_on_image(image, BOX_XYWH))
print("Instances:", len(output["scores"]))
show_sam3_results(image, output)


## 11. Tải kết quả về máy

Nếu bạn vừa chạy lại prompt hoặc box prompt, hãy chạy lại cell xuất ở mục **8** trước, rồi tải ZIP:

In [ ]:
from google.colab import files

# Tạo ZIP mới từ thư mục output hiện tại.
zip_path = shutil.make_archive(OUTPUT_DIR, "zip", OUTPUT_DIR)
files.download(zip_path)


## Ghi chú khi đánh giá

- Mask tốt là mask bao đúng phần artwork, không phải chỉ box bao quanh.
- Soi kỹ vùng viền thêu, sợi tua rua, chữ nhỏ và các vùng gần màu nền.
- `cutout_XX.png` là pixel ảnh gốc với alpha từ mask SAM 3. Nếu chữ/logo bị sai trong file này thì nguyên nhân nằm ở mask; notebook không vẽ lại chi tiết.
- Thử vài cách viết prompt khác nhau. Nếu text prompt không đủ chính xác, dùng box prompt ở mục 10 để xem model có cải thiện không.
- Nếu CUDA OOM, restart runtime rồi chạy lại từng cell; không chạy đồng thời nhiều prompt.

**Tài liệu chính thức:**
- [SAM 3 GitHub](https://github.com/facebookresearch/sam3)
- [SAM 3 image predictor example](https://github.com/facebookresearch/sam3/blob/main/examples/sam3_image_predictor_example.ipynb)
- [SAM 3 gated checkpoint trên Hugging Face](https://huggingface.co/facebook/sam3)
- [Colab past runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html)